# Hugging Face Transformers: Full Learning Practice

This notebook covers 40 stages of learning Hugging Face Transformers, from basics to advanced deployment and capstone projects. We'll provide explanations, simple code, and practical exercises for each topic.

## Stage 1: Introduction
**Topics**: What is Hugging Face, Transformers, Model Hub

In [ ]:
# Stage 1: Introduction
# Hugging Face is the open-source hub for machine learning, providing pre-trained models and datasets.
# The Transformers library provides APIs to easily download and use these models.
# Let's start by looking at what you would normally import.
# Note: Ensure you have `transformers` installed (!pip install transformers)

import transformers
print(f"Transformers version: {transformers.__version__}")
print("You can browse models at: https://huggingface.co/models")

## Stage 2: Installation & Setup
**Topics**: transformers, torch, datasets, verify GPU

In [ ]:
# Stage 2: Installation & Setup
import torch

# Check if CUDA (GPU support) is available
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# A simple tensor operation to verify torch is working
x = torch.tensor([1.0, 2.0, 3.0]).to(device)
print("Tensor on device:", x)

In [2]:
import subprocess

# Stage 2: Installation & Setup - Install required libraries

packages = ["torch", "datasets", "tokenizers", "transformers"]

for package in packages:
    print(f"Installing {package}...")
    subprocess.run(["pip", "install", package], capture_output=True)
    print(f"{package} installed successfully!")

print("\nAll packages installed!")

Installing torch...
torch installed successfully!
Installing datasets...
datasets installed successfully!
Installing tokenizers...
tokenizers installed successfully!
Installing transformers...
transformers installed successfully!

All packages installed!


In [10]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

model_name = "distilbert-base-uncased-finetuned-sst-2-english"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

inputs = tokenizer("I love Hugging Face!", return_tensors="pt")
inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model(**inputs)

print(outputs.logits)

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tensor([[-4.2773,  4.6261]])


## Stage 3: Pipelines
**Topics**: pipeline(), task selection

In [ ]:
# Stage 3: Pipelines
# The pipeline() function is the easiest way to use a pretrained model for inference.
from transformers import pipeline

# 1. Sentiment Analysis
classifier = pipeline("sentiment-analysis")
result = classifier("I absolutely love using Hugging Face!")
print("Sentiment Analysis:", result)

# 2. Text Generation (using a small default model like distilgpt2 or gpt2)
generator = pipeline("text-generation", model="distilgpt2")
gen_result = generator("Once upon a time in a virtual world,", max_length=20, num_return_sequences=1)
print("\nText Generation:", gen_result[0]['generated_text'])

## Stage 4: Tokenizers
**Topics**: Tokenization, Vocabulary, Encoding/Decoding

In [ ]:
# Stage 4: Tokenizers
# Tokenizers convert text into numbers (input IDs) that models can understand.
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

text = "Tokenizing text is fun!"
# Encoding
tokens = tokenizer.tokenize(text)
input_ids = tokenizer.encode(text)

print("Tokens:", tokens)
print("Input IDs:", input_ids)

# Decoding back to text
decoded = tokenizer.decode(input_ids)
print("Decoded text:", decoded)

## Stage 5: Auto Classes
**Topics**: AutoTokenizer, AutoModel

In [ ]:
# Stage 5: Auto Classes
# Auto Classes automatically guess the model architecture from the checkpoint name.
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "distilbert-base-uncased-finetuned-sst-2-english"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

print("Loaded Tokenizer:", type(tokenizer))
print("Loaded Model:", type(model))

# This makes switching models as easy as changing the model_name string!

## Stage 6: Model Outputs
**Topics**: Hidden states, logits, attentions

In [ ]:
# Stage 6: Model Outputs
import torch
from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
# We load the base model (no classification head) to see hidden states
model = AutoModel.from_pretrained("bert-base-uncased")

inputs = tokenizer("Hello, my dog is cute", return_tensors="pt")
outputs = model(**inputs)

# The last hidden state represents the contextual embeddings for each token
last_hidden_states = outputs.last_hidden_state
print("Last hidden state shape:", last_hidden_states.shape) # (batch_size, sequence_length, hidden_size)

## Stage 7: Common NLP Tasks
**Topics**: NER, POS, QA, Summarization

In [ ]:
# Stage 7: Common NLP Tasks
from transformers import pipeline

# 1. Named Entity Recognition (NER)
ner = pipeline("ner", grouped_entities=True)
print("NER:", ner("My name is Sylvain and I work at Hugging Face in Brooklyn."))

# 2. Question Answering
qa = pipeline("question-answering")
context = "Hugging Face was founded in 2016 by Clément Delangue, Julien Chaumond, and Thomas Wolf."
question = "When was Hugging Face founded?"
print("\nQA:", qa(question=question, context=context))

## Stage 8: Text Generation
**Topics**: Sampling, Temperature, Top-k

In [ ]:
# Stage 8: Text Generation Details
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("distilgpt2")
model = AutoModelForCausalLM.from_pretrained("distilgpt2")

inputs = tokenizer("The future of AI is", return_tensors="pt")

# Generate with parameters
outputs = model.generate(
    **inputs,
    max_length=30,
    temperature=0.7, # Lower temperature = more focused/deterministic
    do_sample=True,  # Enable sampling instead of greedy decoding
    top_k=50,        # Filter to top 50 likely next words
    pad_token_id=tokenizer.eos_token_id
)

print("Generated:", tokenizer.decode(outputs[0], skip_special_tokens=True))

## Stage 9: Embeddings
**Topics**: Sentence embeddings, Feature extraction

In [ ]:
# Stage 9: Embeddings
# Embeddings capture the semantic meaning of text.
# For sentence embeddings, 'sentence-transformers' is highly recommended.
# Here's feature extraction using core transformers:

from transformers import pipeline
import numpy as np

# Feature extraction pipeline returns the raw hidden states
extractor = pipeline("feature-extraction", model="bert-base-uncased")
features = extractor("Machine learning is fascinating.", return_tensors="pt")

# The shape is (batch, seq_len, hidden_size)
print("Features shape:", features.shape)
# To get a sentence embedding, we often average the token embeddings (mean pooling)
sentence_embedding = features.mean(dim=1)
print("Sentence embedding shape:", sentence_embedding.shape)

## Stage 10: Datasets Library
**Topics**: Loading datasets, Splits, Mapping

In [ ]:
# Stage 10: Datasets Library
# Hugging Face 'datasets' is heavily optimized for large data.
# !pip install datasets
from datasets import load_dataset

# Load a tiny subset of the rotten_tomatoes dataset
dataset = load_dataset("rotten_tomatoes", split="train[:5]")

print("Dataset features:", dataset.features)
print("\nFirst example:", dataset[0])

# Mapping: Applying a function to every row
def add_length(example):
    example["text_length"] = len(example["text"])
    return example

dataset = dataset.map(add_length)
print("\nFirst example with length:", dataset[0]["text_length"])

## Stage 11: Data Preprocessing
**Topics**: Tokenization pipeline, Dynamic padding

In [ ]:
# Stage 11: Data Preprocessing
from transformers import AutoTokenizer
from datasets import load_dataset

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
dataset = load_dataset("rotten_tomatoes", split="train[:5]")

# Define a tokenization function
def tokenize_function(examples):
    # Padding and truncation prepare sequences to be the same length
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=64)

# Apply to dataset using map (batched=True speeds it up)
tokenized_datasets = dataset.map(tokenize_function, batched=True)

print("Keys after tokenization:", tokenized_datasets.column_names)
print("Input IDs of first example:", tokenized_datasets[0]["input_ids"])

## Stage 12: Fine-Tuning Basics
**Topics**: Fine-tuning vs Pretraining

In [ ]:
# Stage 12: Fine-Tuning Basics
# Fine-tuning adapts a pretrained model to a specific task (like sentiment analysis).
# We take a base model (like BERT) and replace its head with a classification layer.

from transformers import AutoModelForSequenceClassification

# Load model with a classification head for 2 classes
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

print("Model loaded for Sequence Classification.")
# The model now has a `classifier` layer at the top instead of a language modeling head.
# Next stage will show how to train it.

## Stage 13: Trainer API
**Topics**: Trainer, TrainingArguments

In [ ]:
# Stage 13: Trainer API
# The Trainer API abstracts away the training loop.
from transformers import TrainingArguments, Trainer

# Define training arguments (hyperparameters)
training_args = TrainingArguments(
    output_dir="./test_trainer",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    learning_rate=5e-5,
    evaluation_strategy="epoch"
)

print("TrainingArguments configured.")
# To run this, you would pass `model`, `args`, `train_dataset`, `eval_dataset` into Trainer.
# trainer = Trainer(model=model, args=training_args, train_dataset=train_data)
# trainer.train()

## Stage 14: Evaluation
**Topics**: Accuracy, F1, Metrics

In [ ]:
# Stage 14: Evaluation
# Hugging Face provides 'evaluate' library for metrics.
# !pip install evaluate
import evaluate
import numpy as np

metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # Get the index with the highest logit prediction
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

print("Compute metrics function defined. Ready to be passed to Trainer!")

## Stage 15: Saving Models
**Topics**: Save tokenizer, model locally

In [ ]:
# Stage 15: Saving Models
# After training, you must save your model and tokenizer to use them later.

# Assume we have `model` and `tokenizer` loaded from earlier stages
# model.save_pretrained("./my_saved_model")
# tokenizer.save_pretrained("./my_saved_model")

print("Models can be saved using `.save_pretrained('./path')`")
print("You can reload them using `.from_pretrained('./path')`")

## Stage 16: Hugging Face Hub
**Topics**: Upload model, Model Cards

In [ ]:
# Stage 16: Hugging Face Hub
# The Hub is where you share models. You can push directly from code.
from huggingface_hub import notebook_login

print("To upload models, you authenticate first using:")
print("from huggingface_hub import login")
print("login('YOUR_HF_TOKEN')")

print("\nThen use:")
print("model.push_to_hub('your-username/my-awesome-model')")
print("tokenizer.push_to_hub('your-username/my-awesome-model')")

## Stage 17: Custom Datasets
**Topics**: CSV, JSON, Local datasets

In [ ]:
# Stage 17: Custom Datasets
from datasets import load_dataset
import pandas as pd

# Creating a dummy CSV
df = pd.DataFrame({"text": ["I love this", "I hate this"], "label": [1, 0]})
df.to_csv("dummy.csv", index=False)

# Loading custom data
custom_dataset = load_dataset("csv", data_files="dummy.csv")
print("Loaded custom dataset:", custom_dataset)
print("Train split:", custom_dataset['train'][0])

## Stage 18: Custom Tokenizers
**Topics**: Build tokenizer from scratch

In [ ]:
# Stage 18: Custom Tokenizers
# Sometimes you need a tokenizer for a new language or domain.
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import Whitespace

# Initialize a blank Byte-Pair Encoding tokenizer
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Whitespace()

print("Blank BPE Tokenizer created. Ready to be trained on a corpus using:")
print("tokenizer.train(files=['corpus.txt'])")

## Stage 19: Training From Scratch
**Topics**: Build Transformer without pretrained weights

In [ ]:
# Stage 19: Training From Scratch
from transformers import BertConfig, BertModel

# Define custom configuration (e.g., smaller model)
config = BertConfig(
    hidden_size=256,
    num_hidden_layers=4,
    num_attention_heads=4,
    intermediate_size=1024
)

# Initialize model from config (RANDOM WEIGHTS, not pretrained!)
model = BertModel(config)

print("Initialized model with random weights!")
print(f"Layers: {config.num_hidden_layers}, Hidden Size: {config.hidden_size}")

## Stage 20: Optimization
**Topics**: FP16, Gradient Accumulation

In [ ]:
# Stage 20: Optimization
from transformers import TrainingArguments

# Training Arguments optimized for low memory
optimized_args = TrainingArguments(
    output_dir="./opt_trainer",
    per_device_train_batch_size=4,  # Small batch size
    gradient_accumulation_steps=4,  # Accumulate gradients (effective batch size = 4*4=16)
    fp16=True,                      # Use Mixed Precision (16-bit) to save VRAM (requires GPU)
    gradient_checkpointing=True     # Saves memory at the cost of compute speed
)

print("Optimization arguments prepared to reduce memory usage!")

## Stage 21: Accelerate
**Topics**: Multi-GPU, Mixed Precision

In [ ]:
# Stage 21: Accelerate
# Hugging Face 'accelerate' abstracts multi-GPU and mixed precision setups.
# !pip install accelerate

from accelerate import Accelerator

# Initialize accelerator
# accelerator = Accelerator(mixed_precision="fp16")
# device = accelerator.device

# Prepare model, optimizer, dataloader
# model, optimizer, dataloader = accelerator.prepare(model, optimizer, dataloader)

print("Accelerate allows scaling from single GPU to multi-node clusters with minimal code changes.")

## Stage 22: PEFT
**Topics**: LoRA, Adapters

In [ ]:
# Stage 22: PEFT (Parameter-Efficient Fine-Tuning)
# LoRA (Low-Rank Adaptation) freezes the pretrained model weights and injects trainable rank decomposition matrices.
# !pip install peft

print("PEFT allows training Large Language Models (LLMs) on consumer hardware.")
print("Example usage:")
print("from peft import LoraConfig, get_peft_model")
print("config = LoraConfig(r=8, lora_alpha=16, target_modules=['q_proj', 'v_proj'])")
print("peft_model = get_peft_model(model, config)")
print("peft_model.print_trainable_parameters()  # E.g., Trainable params: 0.1%!")

## Stage 23: Quantization
**Topics**: 8-bit, 4-bit, BitsAndBytes

In [ ]:
# Stage 23: Quantization
# Reduces memory footprint of models drastically.
# !pip install bitsandbytes

from transformers import BitsAndBytesConfig

# Config for loading an LLM in 4-bit precision
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype="float16" # compute in fp16
)

print("Quantization config ready.")
print("Use it inside from_pretrained: model = AutoModelForCausalLM.from_pretrained('model_id', quantization_config=quantization_config)")

## Stage 24: Inference Optimization
**Topics**: Flash Attention, KV Cache

In [ ]:
# Stage 24: Inference Optimization
print("To speed up generation with LLMs:")
print("1. Flash Attention v2: Hugging Face supports this via `attn_implementation='flash_attention_2'` in `from_pretrained`.")
print("2. KV Cache: Re-uses Key-Value states from previous tokens. Enabled by default in HF (`use_cache=True`).")
print("3. BetterTransformer: Converts HF models to PyTorch optimized versions.")

## Stage 25: ONNX Export
**Topics**: Export model to ONNX

In [ ]:
# Stage 25: ONNX Export
# ONNX (Open Neural Network Exchange) provides optimized inference across platforms.
# Hugging Face provides 'optimum' for this.

print("To export a model to ONNX:")
print("python -m optimum.exporters.onnx --model distilbert-base-uncased distilbert_onnx/")
print("\nTo load:")
print("from optimum.onnxruntime import ORTModelForSequenceClassification")
print("model = ORTModelForSequenceClassification.from_pretrained('distilbert_onnx/')")

## Stage 26: Torch Compile
**Topics**: torch.compile()

In [ ]:
# Stage 26: Torch Compile
# Introduced in PyTorch 2.0, `torch.compile()` JIT compiles your model for massive speedups.

print("Using torch.compile is extremely easy:")
print("import torch")
print("model = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased')")
print("compiled_model = torch.compile(model)")
print("\nThis optimizes the graph and runs much faster on modern GPUs.")

## Stage 27: Model Configuration
**Topics**: Config files, GenerationConfig

In [ ]:
# Stage 27: Model Configuration
from transformers import AutoConfig, GenerationConfig

# Inspecting architecture config
config = AutoConfig.from_pretrained("gpt2")
print("GPT-2 Config - Vocab Size:", config.vocab_size)
print("GPT-2 Config - Number of Layers:", config.n_layer)

# GenerationConfig
# You can save and load generation presets
gen_config = GenerationConfig(max_new_tokens=50, temperature=0.8, top_p=0.9, do_sample=True)
print("\nGeneration Config defined.")

## Stage 28: Attention Visualization
**Topics**: Attention maps, Hidden layers

In [ ]:
# Stage 28: Attention Visualization
from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
# Enable output_attentions to get the attention weights
model = AutoModel.from_pretrained("bert-base-uncased", output_attentions=True)

inputs = tokenizer("Look at my attention!", return_tensors="pt")
outputs = model(**inputs)

# Attentions is a tuple of tensors (one per layer)
attentions = outputs.attentions
print(f"Number of layers with attention maps: {len(attentions)}")
print(f"Shape of attention in last layer: {attentions[-1].shape}") # (batch, num_heads, seq_len, seq_len)

## Stage 29: Explainability
**Topics**: Explain predictions

In [ ]:
# Stage 29: Explainability
print("Explainability tools help us understand WHY a model made a prediction.")
print("Libraries like 'captum' or 'shap' integrate well with Transformers.")
print("\nExample concept: Integrated Gradients can highlight which input tokens contributed most to the 'Positive Sentiment' output logit.")

## Stage 30: Chat Models
**Topics**: Chat templates, System prompts

In [ ]:
# Stage 30: Chat Models
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("HuggingFaceH4/zephyr-7b-beta")

# Chat models expect conversations formatted in specific ways (User, Assistant, System).
# HF Tokenizers now have `apply_chat_template`.
messages = [
    {"role": "system", "content": "You are a friendly chatbot."},
    {"role": "user", "content": "Hi, how are you?"}
]

# Formatting the prompt
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print("Formatted Chat Prompt:\n", prompt)

## Stage 31: Retrieval-Augmented Generation (RAG)
**Topics**: Embeddings, Vector DB

In [ ]:
# Stage 31: RAG
print("RAG Pipeline Steps:")
print("1. Ingest: Chunk documents and embed them using a Sentence Transformer.")
print("2. Store: Save embeddings in a Vector DB (Chroma, FAISS, Pinecone).")
print("3. Retrieve: Embed user query, search Vector DB for top-k similar chunks.")
print("4. Generate: Pass chunks + query to an LLM context window to generate the final answer.")

## Stage 32: Multimodal Models
**Topics**: Vision Transformers, Image Captioning

In [ ]:
# Stage 32: Multimodal Models
# Hugging Face supports Vision-Language models like BLIP.
from transformers import pipeline

# Note: this requires 'Pillow' (PIL) installed to handle images
print("To caption an image:")
print("captioner = pipeline('image-to-text', model='Salesforce/blip-image-captioning-base')")
print("result = captioner('image.jpg')")
print("print(result)")

## Stage 33: Audio Models
**Topics**: Whisper, Speech Recognition

In [ ]:
# Stage 33: Audio Models
print("Hugging Face makes Audio tasks (ASR) easy with models like OpenAI's Whisper.")
print("\nUsage:")
print("transcriber = pipeline('automatic-speech-recognition', model='openai/whisper-small')")
print("result = transcriber('audio.mp3')")
print("print(result['text'])")

## Stage 34: Vision Models
**Topics**: ViT, Image Classification

In [ ]:
# Stage 34: Vision Models
print("Vision Transformers (ViT) apply the transformer architecture to image patches.")
print("\nUsage:")
print("classifier = pipeline('image-classification', model='google/vit-base-patch16-224')")
print("result = classifier('cat.jpg')")
print("print(result)")

## Stage 35: Diffusion Models
**Topics**: Stable Diffusion, Diffusers

In [ ]:
# Stage 35: Diffusion Models
# The 'diffusers' library by Hugging Face handles image generation.
print("To generate images from text:")
print("from diffusers import StableDiffusionPipeline")
print("pipe = StableDiffusionPipeline.from_pretrained('runwayml/stable-diffusion-v1-5')")
print("pipe = pipe.to('cuda')")
print("image = pipe('a photo of an astronaut riding a horse on mars').images[0]")
print("image.save('astronaut.png')")

## Stage 36: Deployment
**Topics**: FastAPI, Gradio

In [ ]:
# Stage 36: Deployment
print("Gradio is heavily integrated with Hugging Face.")
print("You can build a web UI for your model in 3 lines:")
print("\nimport gradio as gr")
print("from transformers import pipeline")
print("pipe = pipeline('sentiment-analysis')")
print("gr.Interface.from_pipeline(pipe).launch()")

## Stage 37: Production
**Topics**: Inference Endpoints, Docker

In [ ]:
# Stage 37: Production
print("For enterprise production:")
print("1. Hugging Face Inference Endpoints: Deploy models on dedicated managed cloud infrastructure.")
print("2. Text Generation Inference (TGI): HF's highly optimized Docker container for serving LLMs (supports continuous batching, quantization).")

## Stage 38: Monitoring
**Topics**: Logging, Drift

In [ ]:
# Stage 38: Monitoring
print("Once deployed, you must monitor:")
print("- Latency (Time taken to generate)")
print("- Throughput (Tokens per second)")
print("- Data Drift (Are user inputs looking different from training data?)")
print("- Output quality monitoring using LLM-as-a-judge.")

## Stage 39: Security
**Topics**: Prompt Injection, Model Safety

In [ ]:
# Stage 39: Security
print("Security in LLMs is critical.")
print("- Prompt Injection: Users crafting inputs to bypass system rules.")
print("- Data Leakage: Models outputting sensitive training data.")
print("- Content Moderation: Use Guardrail models (e.g., Llama-Guard) to filter bad inputs/outputs.")

## Stage 40: Capstone Projects
**Topics**: End-to-end NLP and LLM applications

In [ ]:
# Stage 40: Capstone Projects
# Ideas to cement your knowledge:
print("Project 1: Fine-tune a specialized summarizer (e.g., legal documents).")
print("Project 2: Build a full RAG Chatbot using an open-source LLM, LangChain, and a Vector DB.")
print("Project 3: Train an Image Captioning app and deploy it on Hugging Face Spaces via Gradio.")
print("\nCongratulations on completing the Hugging Face Transformers track!")